Step-0 : Setup

In [24]:
%%writefile requirements.txt
mcp>=1.0.0
langchain-mcp-adapters
langgraph>=0.2.0
langchain>=0.3.0
langchain-google-genai>=2.0.0
llama-index-core>=0.11.0
llama-index-llms-gemini>=0.3.0
llama-index-embeddings-google
llama-index-readers-file
nest_asyncio
uvicorn
fastapi
python-dotenv
reportlab

Overwriting requirements.txt


In [ ]:
!pip install -r requirements.txt

  Using cached langchain_mcp_adapters-0.1.14-py3-none-any.whl.metadata (10 kB)
  Using cached langchain_google_genai-3.2.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached llama_index_core-0.14.8-py3-none-any.whl.metadata (2.5 kB)
  Using cached llama_index_llms_gemini-0.6.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached llama_index_embeddings_google-0.4.1-py3-none-any.whl.metadata (722 bytes)
  Using cached llama_index_readers_file-0.5.5-py3-none-any.whl.metadata (5.7 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached google_ai_generativelanguage-0.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached banks-2.2.0-py3-none-any.whl.metadata (12 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached deprecated-1.3.1-py2.py3-none-any.whl.metadata (5.9 kB)
  Using cached dirtyjson-1.0.8-py3-none-any.whl.metadata (11 kB)
  Using cached llama_index_workflows-2.11.5-py3-none-any.whl.metadata (4.7 kB)
  Using cached se

In [ ]:
import os
import nest_asyncio
import sys

# Apply nest_asyncio to allow nested event loops in Colab
nest_asyncio.apply()

In [ ]:
import os
from google.colab import userdata

# Option 1: If you saved it in Colab Secrets (Left sidebar > Key icon)
# os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# Option 2: Direct input (Uncomment below if you don't use secrets)
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

if not os.environ.get("GOOGLE_API_KEY"):
    raise ValueError("Please set your GOOGLE_API_KEY")

In [ ]:
# @title 3. Generate Dummy Company Policy PDF (Fixed)
from reportlab.pdfgen import canvas
import os

# Ensure directory exists
os.makedirs("data", exist_ok=True)

pdf_path = "./data/policy.pdf"

# 1. Initialize Canvas with the filename
c = canvas.Canvas(pdf_path)

# 2. Add text
c.drawString(100, 800, "CONFIDENTIAL: TechCorp Bonus Policy 2025")
c.drawString(100, 780, "------------------------------------------")
c.drawString(100, 750, "1. Performance Score Definitions:")
c.drawString(100, 735, "   - Score 5.0: Legendary (25% Bonus)")
c.drawString(100, 720, "   - Score 4.5 to 4.9: High Performer (20% Bonus)")
c.drawString(100, 705, "   - Score 3.0 to 4.4: Meets Expectations (10% Bonus)")
c.drawString(100, 690, "   - Score < 3.0: No Bonus")
c.drawString(100, 660, "2. Eligibility:")
c.drawString(100, 645, "   - Employees must be employed for > 6 months.")

# 3. Save (No arguments needed here)
c.save()

print(f"✅ Created dummy PDF at {pdf_path}")

✅ Created dummy PDF at ./data/policy.pdf


Step-1 : 

In [ ]:
%%writefile knowledge.py
from mcp.server.fastmcp import FastMCP
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.gemini import Gemini
from llama_index.embeddings.google import GooglePaLMEmbedding
import os

# Initialize MCP
mcp = FastMCP("KnowledgeServer")

# Global variables for lazy loading
query_engine = None

def get_engine():
    global query_engine
    if query_engine is None:
        # Setup LlamaIndex with Gemini
        Settings.llm = Gemini(model="models/gemini-1.5-pro")
        Settings.embedding = GooglePaLMEmbedding(model_name="models/embedding-001")
        
        # Load Data
        print("Loading documents...")
        documents = SimpleDirectoryReader("./data").load_data()
        index = VectorStoreIndex.from_documents(documents)
        query_engine = index.as_query_engine()
    return query_engine

@mcp.tool()
def query_policy_documents(query: str) -> str:
    """
    Search the company policy PDF documents.
    Use this to find specific rules about bonuses, holidays, or performance criteria.
    """
    engine = get_engine()
    response = engine.query(query)
    return str(response)

if __name__ == "__main__":
    mcp.run(transport="stdio")

In [ ]:
%%writefile actions.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("ActionServer")

@mcp.tool()
def calculate_final_payout(salary: float, bonus_percentage: float) -> str:
    """
    Calculates the final payout amount.
    args:
        salary: The base annual salary (e.g., 100000)
        bonus_percentage: The percentage as a decimal (e.g., 0.20 for 20%)
    """
    bonus_amount = salary * bonus_percentage
    total = salary + bonus_amount
    return f"Base Salary: ${salary}, Bonus Amount: ${bonus_amount}, Total Payout: ${total}"

if __name__ == "__main__":
    mcp.run(transport="stdio")

In [ ]:
# @title 5. Run the Advanced Agent
import asyncio
import sys
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.messages import HumanMessage

async def run_enterprise_agent():
    # 1. Configuration for connecting to the local files we just made
    server_config = {
        "knowledge": {
            "command": sys.executable, # Use the current Colab python executable
            "args": ["knowledge.py"],
            "transport": "stdio"
        },
        "actions": {
            "command": sys.executable,
            "args": ["actions.py"],
            "transport": "stdio"
        }
    }

    print("🔌 Booting up MCP Servers (Knowledge & Actions)...")
    
    # 2. Connect via MCP Client
    async with MultiServerMCPClient(server_config) as client:
        # Load tools dynamically from the servers
        tools = await client.get_tools()
        print(f"✅ Connected! Found tools: {[t.name for t in tools]}")
        
        # 3. Initialize the Brain (Gemini 1.5 Pro)
        llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0)
        
        # 4. Create the LangGraph Agent
        agent = create_react_agent(llm, tools)
        
        # 5. Define the Scenario
        user_query = (
            "I need to calculate the bonus for an employee named Sarah. "
            "Her base salary is $120,000 and her performance score was 4.8. "
            "First, query the policy documents to find what bonus percentage a score of 4.8 gets. "
            "Then, use that percentage to calculate her final payout."
        )
        
        print(f"\n📝 USER QUERY: {user_query}\n" + "-"*50)
        
        # 6. Run the Graph
        async for chunk in agent.astream(
            {"messages": [HumanMessage(content=user_query)]}, 
            stream_mode="values"
        ):
            # Pretty print the last message from the chunk
            msg = chunk["messages"][-1]
            if msg.type == "ai":
                print(f"\n🤖 AI: {msg.content}")
                if msg.tool_calls:
                    print(f"   🛠️  Calling Tools: {msg.tool_calls}")
            elif msg.type == "tool":
                print(f"   📦 Tool Output: {msg.content}")

# Execute
await run_enterprise_agent()

ModuleNotFoundError: No module named 'langchain_google_genai'